# Phase 3D Colab controller

This notebook is a thin runtime controller. It checks out the frozen repository commit, installs the method environment, verifies the runtime, and invokes `code/benchmarks/run_phase3d.py` for one JSON config. Scientific method logic remains in repository modules.

In [ ]:
import os, platform, subprocess, sys, json
from pathlib import Path

CONFIG_PATH = Path('/content/multiomics-research/07_models/02_classical_integration/configs/EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json')
REPOSITORY = Path('/content/multiomics-research')
EXPECTED_COMMIT = '951a61ca15239a7f644b60ec56dd1a463cd5570f'
print(json.dumps({
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'colab_detected': 'COLAB_RELEASE_TAG' in os.environ,
    'config_path': str(CONFIG_PATH),
    'expected_commit': EXPECTED_COMMIT,
}, indent=2))


In [ ]:
# Clone/check out the exact repository commit. Set REPOSITORY_URL to the approved GitHub mirror.
REPOSITORY_URL = os.environ.get('ASTRA_REPOSITORY_URL', 'https://github.com/rifatahsanpul0k/research.git')
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == EXPECTED_COMMIT, (actual, EXPECTED_COMMIT)
print('checked_out_commit', actual)


In [ ]:
# Install only the environment required by the selected frozen config.
config = json.loads(CONFIG_PATH.read_text())
method = config['method']
if method == 'MOFAPLUS':
    packages = ['numpy==2.4.4', 'scipy==1.17.1', 'scikit-learn==1.9.1', 'h5py==3.16.0', 'mofapy2==0.7.5']
else:
    packages = ['numpy==2.4.4', 'scipy==1.17.1', 'scikit-learn==1.9.1', 'h5py==3.16.0', 'POT==0.9.6.post1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
print(json.dumps({'method': method, 'packages': packages}, indent=2))


In [ ]:
# Execute one frozen config and capture compact output. Dataset access must be mounted separately and read-only.
os.environ['ASTRA_COLAB_CONFIRMED'] = '1'
os.environ.setdefault('COLAB_RUNTIME_TYPE', 'user-selected')
os.environ.setdefault('ASTRA_COLAB_RAM_CLASS', 'record-from-runtime')
result = subprocess.run(
    [sys.executable, str(REPOSITORY / 'code/benchmarks/run_phase3d.py'), '--config', str(CONFIG_PATH)],
    cwd=REPOSITORY, text=True, capture_output=True, check=False,
)
print(result.stdout)
if result.returncode:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f'Phase 3D run failed with exit code {result.returncode}')
print('Phase 3D run completed; review the saved experiment directory and provenance locally.')
